# Hallucination Detection — Colab Experiments

This notebook runs the NLI, BERT, RoBERTa, DeBERTa, or SciBERT experiment using the same context-grouped evaluation protocol.

Before starting, select **Runtime → Change runtime type → T4 GPU**. Run the cells from top to bottom. Run one transformer model at a time because each model trains six grouped folds.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Enable a GPU from Runtime > Change runtime type before continuing.')

## Download the project and install its dependencies

In [ ]:
REPOSITORY = 'https://github.com/Wadha667/llm-hallucination-detection.git'
BRANCH = 'research-and-model-evaluation'
PROJECT_DIR = Path('/content/llm-hallucination-detection')

if PROJECT_DIR.exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPOSITORY, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Project ready at', PROJECT_DIR)

## Save results to Google Drive

The output is saved in Drive so it remains available if the Colab session disconnects.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
DRIVE_OUTPUT = Path('/content/drive/MyDrive/llm-hallucination-results')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
print('Results will be saved to', DRIVE_OUTPUT)

## Choose and run an experiment

Select **nli** for the fastest first experiment. Then run BERT, RoBERTa, DeBERTa, and SciBERT in separate Colab sessions. Do not report a transformer result unless all six folds finish.

In [ ]:
MODEL = 'nli'  # @param ['nli', 'bert', 'roberta', 'deberta', 'scibert']
BATCH_SIZE = 8  # @param {type:'integer'}
EPOCHS = 5  # @param {type:'integer'}
SEED = 42  # @param {type:'integer'}

from pathlib import Path

if 'DRIVE_OUTPUT' not in globals():
    DRIVE_OUTPUT = Path('/content/llm-hallucination-results')
    DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
    print('Google Drive was not mounted. Results will use temporary Colab storage:', DRIVE_OUTPUT)

if not Path('experiments/run_nli_baseline.py').exists():
    raise RuntimeError('Run the project download and installation cell before this cell.')

run_output = DRIVE_OUTPUT / MODEL
run_output.mkdir(parents=True, exist_ok=True)

if MODEL != 'nli':
    gpu_test = subprocess.run(
        [sys.executable, '-c', "import torch; print(torch.cuda.is_available())"],
        capture_output=True, text=True, check=True,
    )
    if gpu_test.stdout.strip() != 'True':
        raise RuntimeError(
            'No GPU is active. In Colab choose Runtime > Change runtime type > T4 GPU, '
            'then restart and run the notebook from the top.'
        )

if MODEL == 'nli':
    command = [
        sys.executable, 'experiments/run_nli_baseline.py',
        '--output-dir', str(run_output),
        '--batch-size', str(BATCH_SIZE),
    ]
else:
    command = [
        sys.executable, 'experiments/run_transformer_cv.py',
        '--model', MODEL,
        '--output-dir', str(run_output),
        '--batch-size', str(BATCH_SIZE),
        '--epochs', str(EPOCHS),
        '--seed', str(SEED),
    ]

print('Running:', ' '.join(command))
subprocess.run(command, check=True)

## Review the result

In [ ]:
import pandas as pd

summary = pd.read_csv(run_output / 'summary.csv')
display(summary[summary['metric'].isin(['weighted_f1', 'macro_f1', 'accuracy'])])

folds = pd.read_csv(run_output / 'fold_metrics.csv')
display(folds[['fold', 'weighted_f1', 'macro_f1', 'accuracy']])

## Generate error analysis

In [ ]:
error_output = DRIVE_OUTPUT / f'{MODEL}_error_analysis'
subprocess.run([
    sys.executable, 'experiments/analyze_errors.py',
    '--predictions', str(run_output / 'oof_predictions.csv'),
    '--output-dir', str(error_output),
], check=True)

print((error_output / 'error_analysis.md').read_text(encoding='utf-8')[:5000])

## Download a copy of the results

The original files remain in Google Drive. This cell also creates a ZIP and downloads it to your computer. Upload that ZIP to the project chat so the results can be checked and documented before the next commit.

In [ ]:
import shutil
from google.colab import files

archive_base = Path('/content') / f'{MODEL}_hallucination_results'
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=run_output)
print('Created', archive_path)
files.download(archive_path)

## Experiment checklist

Before adding the result to the report:

- Confirm that six fold rows appear for a fine-tuned transformer.
- Keep `summary.csv`, `fold_metrics.csv`, `oof_predictions.csv`, and `run_config.json`.
- Review the error-analysis files.
- Record failed runs instead of deleting them.
- Do not compare these grouped results directly with the old random-split notebook scores.